# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [4]:

# Task 1 — Fossil fuel dependency treemap

df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

country_fossil = (
    df_fossil.groupby(['Region', 'Country'], as_index=False)['TWh']
    .sum()
)

top_country = country_fossil.loc[country_fossil['TWh'].idxmax()]

region_fossil = (
    df_fossil.groupby('Region')['TWh']
    .sum()
)

top_region = region_fossil.idxmax()

fossil_colors = {
    'Coal': '#D55E00',
    'Oil': '#0072B2',
    'Natural Gas': '#E69F00'
}

fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title=f'{top_region} Leads in Fossil Fuel Consumption'
)

trace = fig.data[0]
labels = list(trace.labels)
ids = list(trace.ids)

node_colors = [
    fossil_colors.get(label, '#BDBDBD')
    if len(node_id.split('/')) == 3
    else '#BDBDBD'
    for label, node_id in zip(labels, ids)
]

trace.marker.colors = node_colors

fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    textinfo='label+value',
    root_color='#BDBDBD'
)

fig.update_layout(
    title_x=0.5,
    margin=dict(t=70, l=10, r=10, b=10)
)

fig.show()

print(f"Highest fossil-consuming country: {top_country['Country']}")
print(f"Highest fossil-consuming region: {top_region}")


Highest fossil-consuming country: Saudi Arabia
Highest fossil-consuming region: Asia


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:

# Task 2 — Low-carbon energy by country

df_lowcarbon = df.loc[
    df['Source_Type'] == 'Low-carbon'
].copy()

low_carbon_country = (
    df_lowcarbon.groupby('Country', as_index=False)['TWh']
    .sum()
)

low_carbon_country['All'] = 'Low-carbon'

top_lowcarbon = low_carbon_country.loc[
    low_carbon_country['TWh'].idxmax()
]

fig1 = px.treemap(
    low_carbon_country,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-Carbon Energy Distribution Across Countries'
)

fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh'
)

fig1.update_layout(
    title_x=0.5,
    coloraxis_colorbar_title='TWh'
)

fig1.show()



low_carbon_sorted = low_carbon_country.sort_values(
    'TWh', ascending=True
)

fig2 = px.bar(
    low_carbon_sorted,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#0072B2'],
    text='TWh',
    title=f"{top_lowcarbon['Country']} Leads in Low-Carbon Energy Production"
)

fig2.update_traces(
    texttemplate='%{text:,.0f} TWh',
    textposition='outside'
)

fig2.update_layout(
    title_x=0.5,
    xaxis_title='Energy (TWh)',
    yaxis_title='Country',
    showlegend=False
)

fig2.show()

print(
    f"Leading country: {top_lowcarbon['Country']} "
    f"({top_lowcarbon['TWh']:,.1f} TWh)"
)


Leading country: France (7,593.0 TWh)


### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

The horizontal bar chart tells the ranking story more clearly because all countries share a common baseline, making it easier to compare values and identify which country produces the most low-carbon energy. The treemap is better for showing the relative contribution of each country and the overall distribution, but comparing similarly sized rectangles is more difficult.

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [6]:

# Task 3 — Regional fossil energy composition

region_source = (
    df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh']
    .sum()
)

region_source['Region_Total'] = (
    region_source.groupby('Region')['TWh'].transform('sum')
)

region_source['Share'] = (
    region_source['TWh'] / region_source['Region_Total']
)

coal_share = (
    region_source[region_source['Source'] == 'Coal']
    [['Region', 'Share']]
    .sort_values('Share', ascending=False)
)

top_coal_region = coal_share.iloc[0]

region_order = coal_share.sort_values(
    'Share', ascending=True
)['Region'].tolist()

region_source['Coal_Label'] = np.where(
    region_source['Source'] == 'Coal',
    region_source['Share'].apply(lambda x: f'{x:.1%}'),
    ''
)

source_colors = {
    'Coal': '#D55E00',
    'Oil': '#BDBDBD',
    'Natural Gas': '#696969'
}

fig = px.bar(
    region_source,
    x='Share',
    y='Region',
    color='Source',
    orientation='h',
    barmode='stack',
    text='Coal_Label',
    color_discrete_map=source_colors,
    category_orders={
        'Source': ['Coal', 'Oil', 'Natural Gas'],
        'Region': region_order
    },
    title=(
        f"{top_coal_region['Region']} is the Most Coal-Dependent "
        f"Region ({top_coal_region['Share']:.1%})"
    )
)

fig.update_traces(
    textposition='inside',
    texttemplate='%{text}'
)

fig.update_layout(
    title_x=0.5,
    xaxis_title='Share of Regional Fossil Energy',
    yaxis_title='Region',
    xaxis=dict(
        tickformat='.0%',
        range=[0, 1]
    ),
    legend_title='Fossil Source'
)

fig.show()

print(
    f"Most coal-dependent region: {top_coal_region['Region']}"
)
print(
    f"Coal share: {top_coal_region['Share']:.1%}"
)


Most coal-dependent region: Africa
Coal share: 78.0%


### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

The treemap answers which countries and regions consume the most fossil fuel energy and shows the hierarchical distribution of consumption across different sources.

The 100% stacked bar chart answers which region depends most heavily on coal relative to its total fossil fuel consumption. It makes comparing the proportions of Coal, Oil and Natural Gas between regions easier.